In [1]:
pip install pybaseball pandas requests beautifulsoup4 pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from pybaseball import statcast_batter, playerid_lookup
import pandas as pd

pd.set_option('display.max_columns', None)

player_id = 663538
game_date = '2026-04-15'

df = statcast_batter(game_date, game_date, player_id)
print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
print(df.columns.tolist())

Gathering Player Data
Shape: (13, 118)
Columns (118):
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'wo

In [2]:
swings = df[df['description'].isin(['swinging_strike', 'foul', 'hit_into_play', 'foul_tip', 'swinging_strike_blocked'])]
print(f"Swings: {len(swings)}, with bat speed: {swings['bat_speed'].notna().sum()}")

Swings: 5, with bat speed: 5


In [3]:
print("description values in this game:")
print(df['description'].value_counts(dropna=False))
print("\nevents values in this game:")
print(df['events'].value_counts(dropna=False))

description values in this game:
description
hit_into_play    5
ball             5
called_strike    4
foul_bunt        1
Name: count, dtype: int64

events values in this game:
events
NaN          10
single        2
field_out     2
triple        1
Name: count, dtype: int64


In [4]:
season_df = statcast_batter('2025-04-01', '2025-09-30', player_id)
print("All description values across season:")
print(season_df['description'].value_counts(dropna=False))
print("\nAll events values across season:")
print(season_df['events'].value_counts(dropna=False))

Gathering Player Data
All description values across season:
description
ball                       657
hit_into_play              542
called_strike              434
foul                       427
swinging_strike            101
blocked_ball                58
foul_tip                    12
swinging_strike_blocked     11
hit_by_pitch                 7
foul_bunt                    5
automatic_ball               4
Name: count, dtype: int64

All events values across season:
events
NaN                          1621
field_out                     325
single                        134
strikeout                      49
walk                           36
double                         29
force_out                      24
home_run                        7
hit_by_pitch                    7
sac_fly                         5
triple                          4
grounded_into_double_play       4
field_error                     3
double_play                     3
fielders_choice_out             2
fielders_c

In [5]:
print(df[['at_bat_number', 'pitch_number', 'inning', 'description', 'events']].sort_values(['at_bat_number', 'pitch_number']))

    at_bat_number  pitch_number  inning    description     events
9               9             1       2           ball        NaN
8               9             2       2  called_strike        NaN
7               9             3       2           ball        NaN
6               9             4       2  hit_into_play     single
5              28             1       4           ball        NaN
4              28             2       4  called_strike        NaN
3              28             3       4  hit_into_play  field_out
2              46             1       6      foul_bunt        NaN
1              46             2       6  called_strike        NaN
0              46             3       6  hit_into_play     single
11             58             1       7  called_strike        NaN
10             58             2       7  hit_into_play  field_out
14             76             1      10           ball        NaN
13             76             2      10           ball        NaN
12        

In [11]:
print(df[['description', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom']].head(30))

      description  estimated_woba_using_speedangle  woba_value  woba_denom
0   hit_into_play                            0.756         2.0         1.0
1   hit_into_play                            0.888         0.9         1.0
2   called_strike                              NaN         NaN         NaN
3            ball                              NaN         NaN         NaN
4   hit_into_play                            0.156         0.0         1.0
5            ball                              NaN         NaN         NaN
6   hit_into_play                            0.348         0.9         1.0
7   called_strike                              NaN         NaN         NaN
8            ball                              NaN         NaN         NaN
9   hit_into_play                            0.736         0.0         1.0
10           ball                              NaN         NaN         NaN
11  called_strike                              NaN         NaN         NaN
12           ball        

In [7]:
from pybaseball import statcast_pitcher

pitcher_id = 694973
pitcher_2026 = statcast_pitcher('2026-04-01', '2026-04-29', pitcher_id)
pitcher_2025 = statcast_pitcher('2025-04-01', '2025-09-30', pitcher_id)

print(f"2026 pitches: {len(pitcher_2026)}, 2025 pitches: {len(pitcher_2025)}")
print(f"2026 pitch types: {pitcher_2026['pitch_type'].value_counts()}")

Gathering Player Data
Gathering Player Data
2026 pitches: 409, 2025 pitches: 2903
2026 pitch types: pitch_type
FF    152
CH     66
ST     66
SI     65
FS     51
CU      9
Name: count, dtype: int64


In [8]:
import requests
import io

url = "https://baseballsavant.mlb.com/statcast_search/csv?all=true&hfPT=&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones=&hfPull=&hfC=&hfSea=2025%7C&hfSit=&player_type=batter&hfOuts=&hfOpponent=&pitcher_throws=&batter_stands=&hfSA=&game_date_gt=2026-04-15&game_date_lt=2026-04-15&hfMo=&hfTeam=&home_road=&hfRO=&position=&hfInfield=&team=&hfOutfield=&hfInn=&hfBBT=&batters_lookup%5B%5D=663538&hfFlag=&metric_1=&group_by=name&min_pitches=0&min_results=0&min_pas=0&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc&type=details"
resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
savant_df = pd.read_csv(io.StringIO(resp.text))
print(savant_df.columns.tolist())
print(savant_df.shape)


['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'babip_value', 'iso_value', 'launch_speed_a

In [9]:
print("Dupes:", df.duplicated(subset=['game_pk', 'at_bat_number', 'pitch_number']).sum())
print("Pitch types:", df['pitch_type'].value_counts(dropna=False))
print("Unique batters:", df['batter'].nunique())

Dupes: 0
Pitch types: pitch_type
FF    5
ST    3
SI    3
FC    2
SL    2
Name: count, dtype: int64
Unique batters: 1
